In [0]:
from pyspark.sql import functions as F

# ==========================================
# CONFIGURACIÓN
# ==========================================
storage_account = "adsldevcrm"
container_name = "raw"
scope = "databriksScope"

client_id = dbutils.secrets.get(scope=scope, key="sp-client-id")
client_secret = dbutils.secrets.get(scope=scope, key="sp-client-secret")
tenant_id = dbutils.secrets.get(scope=scope, key="sp-tenant-id")

raw_input_path = f"abfss://{container_name}@{storage_account}.dfs.core.windows.net/data"
bronze_path = f"abfss://{container_name}@{storage_account}.dfs.core.windows.net/bronze"

database_name = "crm_sales_db"

In [0]:
# ==========================================
# AUTENTICACIÓN
# ==========================================
spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", client_secret)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)

In [0]:
# ==========================================
# CREAR DATABASE
# ==========================================
spark.sql(f"CREATE DATABASE IF NOT EXISTS {database_name}")
spark.sql(f"USE {database_name}")

# ==========================================
# VER ARCHIVOS DE ENTRADA
# ==========================================
display(dbutils.fs.ls(raw_input_path))

In [0]:
# ==========================================
# LEER CSV
# ==========================================
accounts_df = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{raw_input_path}/accounts.csv")
data_dictionary_df = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{raw_input_path}/data_dictionary.csv")
products_df = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{raw_input_path}/products.csv")
sales_pipeline_df = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{raw_input_path}/sales_pipeline.csv")
sales_teams_df = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{raw_input_path}/sales_teams.csv")


In [0]:
# ==========================================
# NORMALIZAR NOMBRES DE COLUMNAS
# ==========================================
for c in accounts_df.columns:
    accounts_df = accounts_df.withColumnRenamed(c, c.strip().lower().replace(" ", "_").replace("-", "_").replace(".", "_"))

for c in data_dictionary_df.columns:
    data_dictionary_df = data_dictionary_df.withColumnRenamed(c, c.strip().lower().replace(" ", "_").replace("-", "_").replace(".", "_"))

for c in products_df.columns:
    products_df = products_df.withColumnRenamed(c, c.strip().lower().replace(" ", "_").replace("-", "_").replace(".", "_"))

for c in sales_pipeline_df.columns:
    sales_pipeline_df = sales_pipeline_df.withColumnRenamed(c, c.strip().lower().replace(" ", "_").replace("-", "_").replace(".", "_"))

for c in sales_teams_df.columns:
    sales_teams_df = sales_teams_df.withColumnRenamed(c, c.strip().lower().replace(" ", "_").replace("-", "_").replace(".", "_"))


In [0]:
# ==========================================
# AJUSTES DE COLUMNAS
# ==========================================
if "subsidiary_of" in accounts_df.columns:
    accounts_df = accounts_df.withColumnRenamed("subsidiary_of", "parent_company")

if "table_" in data_dictionary_df.columns:
    data_dictionary_df = data_dictionary_df.withColumnRenamed("table_", "table")

In [0]:
# ==========================================
# AGREGAR COLUMNAS DE AUDITORÍA
# ==========================================
accounts_df = accounts_df.withColumn("_source_file", F.lit("accounts.csv")).withColumn("_ingestion_ts", F.current_timestamp())
data_dictionary_df = data_dictionary_df.withColumn("_source_file", F.lit("data_dictionary.csv")).withColumn("_ingestion_ts", F.current_timestamp())
products_df = products_df.withColumn("_source_file", F.lit("products.csv")).withColumn("_ingestion_ts", F.current_timestamp())
sales_pipeline_df = sales_pipeline_df.withColumn("_source_file", F.lit("sales_pipeline.csv")).withColumn("_ingestion_ts", F.current_timestamp())
sales_teams_df = sales_teams_df.withColumn("_source_file", F.lit("sales_teams.csv")).withColumn("_ingestion_ts", F.current_timestamp())


In [0]:
# ==========================================
# REVISIÓN
# ==========================================
display(accounts_df)
display(data_dictionary_df)
display(products_df)
display(sales_pipeline_df)
display(sales_teams_df)

In [0]:
# ==========================================
# GUARDAR COMO TABLAS BRONZE
# ==========================================
accounts_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{database_name}.bronze_accounts")
data_dictionary_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{database_name}.bronze_data_dictionary")
products_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{database_name}.bronze_products")
sales_pipeline_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{database_name}.bronze_sales_pipeline")
sales_teams_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{database_name}.bronze_sales_teams")


In [0]:
# ==========================================
# GUARDAR RUTAS BRONZE
# ==========================================
accounts_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{bronze_path}/accounts")
data_dictionary_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{bronze_path}/data_dictionary")
products_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{bronze_path}/products")
sales_pipeline_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{bronze_path}/sales_pipeline")
sales_teams_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{bronze_path}/sales_teams")

print("Bronze terminado")